In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Configuration
bookings_stage = "`zoomcarprocessingpipeline`.default.bookings_stage"
customers_stage = "`zoomcarprocessingpipeline`.default.customers_stage"

bookings_target = "`zoomcarprocessingpipeline`.default.booking_target"
customers_target = "`zoomcarprocessingpipeline`.default.customers_target"

print("Starting comprehensive data validation process...")

In [0]:
# Read all staging tables
try:
    df_bookings = spark.read.table(bookings_stage)
    df_customers = spark.read.table(customers_stage)

    print("Successfully loaded all staging tables")
    print(f"Orders: {df_bookings.count()} records")
    print(f"Customers: {df_customers.count()} records")

except Exception as e:
    print(f"Error loading staging tables: {str(e)}")
    raise

In [0]:
#Bookings Data Transformations:
# Extract Start Date & Time
transformed_booking_df = (df_bookings
                  .withColumn("start_date",F.to_date("start_time"))
                  .withColumn("start_time_only",F.date_format("start_time", "HH:mm:ss"))
)

# Extract End Date & Time
transformed_booking_df = (transformed_booking_df
                  .withColumn("end_date", F.to_date("end_time"))
                  .withColumn("end_time_only",F.date_format("end_time", "HH:mm:ss"))
)

# Calculate Total Duration in Hours
transformed_booking_df = transformed_booking_df.withColumn("booking_duration_hours",
                (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 3600
)
transformed_booking_df.show()

In [0]:
# Merge Orders Data with SCD2 Logic
try:

    # Check if target table exists
    if spark.catalog.tableExists(bookings_target):
        # Perform SCD2 merge
        target = DeltaTable.forName(spark, bookings_target)
        target.alias("t").merge(transformed_booking_df.alias("b"),"t.booking_id = b.booking_id").whenMatchedDelete(condition="b.status = 'cancelled'").whenMatchedUpdate( condition="b.status != 'cancelled'",
            set={
            "booking_id": "b.booking_id",
            "customer_id": "b.customer_id",
            "car_id": "b.car_id",
            "booking_date": "b.booking_date",
            "start_time": "b.start_time",
            "end_time": "b.end_time",
            "total_amount": "b.total_amount",
            "status": "b.status",
            "start_time_only": "b.start_time_only",
            "end_time_only": "b.end_time_only",
            "booking_duration_hours": "b.booking_duration_hours"
            }
        )

        # Insert new records
        transformed_booking_df.write.format("delta").mode("append").saveAsTable(bookings_target)
    else:
        # Create new table
        transformed_booking_df.write.format("delta").saveAsTable(bookings_target)
    
    print("Bookings merge completed successfully")
    
except Exception as e:
    print(f"Error merging orders data: {str(e)}")
    raise

spark.table(bookings_target).show()

In [0]:
#Customers Data Transformations:

# Normalize Phone Numbers
# Keep only digits (removes +, -, spaces, brackets, etc.)
transformed_customers_df = df_customers.withColumn("phone_number",F.regexp_replace("phone_number", r"[^0-9]", ""))


# Calculate Customer Tenure
transformed_customers_df = transformed_customers_df.withColumn("customer_tenure_days",
                F.datediff(F.current_date(), F.col("signup_date"))
)

transformed_customers_df.show()



In [0]:
# Merge Orders Data with SCD2 Logic
try:

    # Check if target table exists
    if spark.catalog.tableExists(customers_target):
        # Perform SCD2 merge
        target = DeltaTable.forName(spark, customers_target)
        target.alias("t").merge(transformed_customers_df.alias("c"),"t.customer_id = c.customer_id").whenMatchedUpdate( condition="b.customer_id != ''",
            set={
            "customer_id": "c.customer_id",
            "name": "c.name",
            "email": "c.email",
            "phone_number": "c.phone_number",
            "signup_date": "c.signup_date",
            "status": "c.status",
            "customer_tenure_days": "c.customer_tenure_days"
            }
        )

        # Insert new records
        transformed_customers_df.write.format("delta").mode("append").saveAsTable(customers_target)
    else:
        # Create new table
        transformed_customers_df.write.format("delta").saveAsTable(customers_target)
    
    print("Customers merge completed successfully")
    
except Exception as e:
    print(f"Error merging orders data: {str(e)}")
    raise

spark.table(customers_target).show()